# Shimba LLM — Google Colab

Train a decoder-only transformer from scratch on any text, on Colab's GPU.

**Before you start:** pick `Runtime > Change runtime type > T4 GPU`. Everything
still runs on CPU without it, just much slower.

How it works:

1. Clone the repo and install PyTorch
2. Point it at a text corpus (upload, Google Drive, or a URL)
3. Train — checkpoints save every iteration, so a disconnect is recoverable
4. Generate, chat, and download the model

> Colab resets the filesystem when a runtime is deleted. Mount Drive (step 4)
> and your corpus plus checkpoints will still be there next time.

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────
import os, sys, subprocess, shutil

REPO = "https://github.com/Shimba-crypto/Shimba-LLM-Training.git"
WORKDIR = "/content/shimba-llm"

if os.path.isdir(os.path.join(WORKDIR, "llm")):
    print("Repo already present — pulling latest changes")
    subprocess.run(["git", "-C", WORKDIR, "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO, WORKDIR], check=True)

os.chdir(WORKDIR)
sys.path.insert(0, WORKDIR)
print("Working directory:", os.getcwd())

In [ ]:
# Colab ships CUDA-matched PyTorch already. Installing with --index-url picks up
# the wheel that matches the runtime's CUDA, which is faster than the default.
!pip install -q torch --index-url https://download.pytorch.org/whl/cu124 2>/dev/null || \
  !pip install -q "torch>=2.0.0"
!pip install -q -r requirements.txt

import torch
print("PyTorch:", torch.__version__)

## 1. Confirm the GPU

Colab hands you a T4 by default. You should see `cuda: 0`.

In [ ]:
import torch
from llm.device import resolve_device, describe_device

DEVICE = resolve_device("auto")
print("Device:", describe_device(DEVICE))

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM free: {free/1024**3:.1f} GB / {total/1024**3:.1f} GB")
else:
    print("No GPU found — training will be slow. Check Runtime > Change runtime type.")

In [ ]:
# Sanity check: tokenizer + model + a 5-iteration training loop.
!python quick_test.py

## 2. Mount Google Drive (recommended)

Your corpus and checkpoints persist here, so a disconnected or deleted
runtime doesn't lose your work.

Skip this if you just want a quick test — but then anything you train dies
when the runtime ends.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
print("Drive mounted")

## 3. Configure

Edit `CORPUS_PATH` to point at your text. A `.txt` file, or a **folder** of
`.txt` files which get merged in sorted order.

Not sure what to use? Leave `CORPUS_PATH = None` and the next cell generates
a small sample corpus so you can confirm the whole pipeline works first.

In [ ]:
# ── Edit these ───────────────────────────────────────────────────────
REPO_DIR   = "/content/shimba-llm"
DATA_DIR   = "/content/drive/MyDrive/shimba"     # corpus + checkpoints live here
RUN_NAME   = "shimba_run1"                       # change to start a fresh model
CORPUS_PATH = None                               # e.g. DATA_DIR + "/corpus.txt"

# Model size — bigger = slower, more capacity.
#   tiny   ~0.4M params   fast, good for a first run
#   small ~10M params     the README default
#   medium ~85M params    needs a bigger corpus to be worth it
PRESET = "small"

PRESETS = {
    "tiny":   dict(n_embd=64,  n_layer=2, n_head=2,  block_size=128, batch_size=32, grad_accum=1),
    "small":  dict(n_embd=256, n_layer=4, n_head=4,  block_size=256, batch_size=16, grad_accum=2),
    "medium": dict(n_embd=512, n_layer=6, n_head=8,  block_size=256, batch_size=8,  grad_accum=4),
}

MODEL_PATH = f"{DATA_DIR}/{RUN_NAME}.pth"

CFG = dict(
    **PRESETS[PRESET],
    max_iters=3000,
    lr=3e-4,
    eval_interval=250,
    eval_iters=40,
    amp=True,            # mixed precision — near-free speedup on a T4
)

for k, v in CFG.items():
    print(f"  {k:16s} {v}")
print(f"\n  model out -> {MODEL_PATH}")

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
print("Data dir ready:", DATA_DIR)

## 4. Get a corpus

Pick **one** of the three options below.

**A — Google Drive:** drop a `.txt` (or a folder of them) into `shimba/` in
Drive, then set `CORPUS_PATH` above.

**B — Upload:** run the next cell and choose a file from your computer.

**C — URL:** for public domain books, Project Gutenberg is a good source.

In [ ]:
# ── B: upload from your computer ─────────────────────────────────────
from google.colab import files

CORPUS_PATH = f"{DATA_DIR}/corpus.txt"
uploaded = files.upload()

if uploaded:
    # Notebook stores bytes keyed by filename; write them to Drive.
    with open(CORPUS_PATH, "wb") as f:
        for name, data in uploaded.items():
            f.write(data)
            print("wrote", name, len(data), "bytes")
    print("CORPUS_PATH =", CORPUS_PATH)
else:
    print("Nothing uploaded — keeping CORPUS_PATH as configured.")

In [ ]:
# ── C: download from a URL (optional) ────────────────────────────────
# Example: Moby Dick, ~1.2 MB of text.
# URL = "https://www.gutenberg.org/files/2701/2701-0.txt"

URL = None
if URL:
    import urllib.request
    CORPUS_PATH = f"{DATA_DIR}/corpus.txt"
    urllib.request.urlretrieve(URL, CORPUS_PATH)
    print("downloaded ->", CORPUS_PATH)

In [ ]:
# ── A: sample corpus, so you can test the pipeline with zero setup ───
if CORPUS_PATH is None or not os.path.exists(CORPUS_PATH):
    print("No corpus configured — generating a sample so you can test the flow.")
    import random
    random.seed(0)
    words = ("the quick brown fox jumps over the lazy dog while shimba trains a "
             "small transformer on plain text and learns the shape of language "
             "one character at a time until the loss stops moving").split()
    lines = [" ".join(random.choices(words, k=random.randint(6, 16))).capitalize() + "."
             for _ in range(6000)]
    CORPUS_PATH = f"{DATA_DIR}/sample_corpus.txt"
    with open(CORPUS_PATH, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print("sample corpus ->", CORPUS_PATH)

size_mb = os.path.getsize(CORPUS_PATH) / 1024**2
print(f"Corpus: {CORPUS_PATH}  ({size_mb:.2f} MB)")

## 5. Train

`quick_train.py` writes a checkpoint every iteration and records the
optimizer state, so **Ctrl+C is safe** — the next run with `--resume` picks up
at the exact iteration it stopped.

That matters on Colab: free runtimes get killed without warning.

First run? Lower `CFG["max_iters"]` to `300` and confirm the loss is dropping
before committing to a long one.

In [ ]:
# Build the command from CFG so the two can't drift apart.
cmd = [
    "python", "-u", "quick_train.py",
    "--data",  CORPUS_PATH,
    "--out",   MODEL_PATH,
    "--device", "cuda" if DEVICE.type == "cuda" else "cpu",
    "--max_iters",    str(CFG["max_iters"]),
    "--lr",           str(CFG["lr"]),
    "--n_embd",       str(CFG["n_embd"]),
    "--n_layer",      str(CFG["n_layer"]),
    "--n_head",       str(CFG["n_head"]),
    "--block_size",   str(CFG["block_size"]),
    "--batch_size",   str(CFG["batch_size"]),
    "--grad_accum",   str(CFG["grad_accum"]),
    "--eval_interval", str(CFG["eval_interval"]),
    "--eval_iters",   str(CFG["eval_iters"]),
    "--save_every",   "50",
]
if CFG["amp"]:
    cmd.append("--amp")
if os.path.exists(MODEL_PATH):
    cmd.append("--resume")

print(" ".join(cmd), "\n")
subprocess.run(cmd, check=True)

**Interrupted?** Re-run the cell above — `--resume` is added automatically
whenever `MODEL_PATH` already exists.

Loss should trend downward. It won't be smooth: batches are sampled randomly,
so the number bounces. Judge it by the `val=` column at each eval.

## 6. Generate

A model this size produces fluent-looking fragments, not coherent prose.
Lower `temperature` for repetitive-but-plausible text, raise it for noise.

In [ ]:
import torch
from llm.model import GPT
from llm.tokenizer import CharTokenizer
from llm.generate import generate

tok = CharTokenizer.load(CharTokenizer.default_path(MODEL_PATH))
model = GPT.load(MODEL_PATH, device=DEVICE)

params = sum(p.numel() for p in model.parameters())
print(f"{params:,} parameters | vocab {tok.vocab_size} | block {model.cfg.block_size}\n")

for prompt in ["The", "Once upon a time", "In the beginning"]:
    print(f"--- {prompt!r} ---")
    print(generate(model, tok, prompt, max_new_tokens=300, temperature=0.7))
    print()

## 7. Chat

A character-level model has no notion of a chat format — this is just the
`User:/Assistant:` prompt wrapper from `chat.py`, and expect it to wander.
Useful as a demo, not as an assistant.

In [ ]:
def chat(prompt, max_tokens=300, temperature=0.7, top_k=40):
    text = generate(model, tok, prompt, max_new_tokens=max_tokens,
                    temperature=temperature, top_k=top_k, top_p=0.95)
    return text[len(prompt):].lstrip()

def converse(user_input, history=None, temperature=0.7):
    history = history if history is not None else []
    prompt = "\n".join(
        [f"User: {u}" if role == "user" else f"Assistant: {a}" for role, u in history]
    ) + f"\nUser: {user_input}\nAssistant:"
    reply = chat(prompt, temperature=temperature)
    history.extend([("user", user_input), ("assistant", reply)])
    return reply, history

# Non-interactive example:
reply, history = converse("Hello, who are you?")
print(reply)

In [ ]:
# Interactive loop — comment this cell out to skip it.
history = []
try:
    while True:
        line = input(">>> ").strip()
        if line in ("/exit", "/quit", ""):
            break
        if line == "/reset":
            history = []
            print("[reset]")
            continue
        reply, history = converse(line, history)
        print("\n" + reply + "\n")
except (EOFError, KeyboardInterrupt):
    print("\nbye")

## 8. Save

Two `.pth` files and one tokenizer — **all three are needed**. The tokenizer
is derived from your corpus, so a model without it can't be loaded.

Keep them together in the same folder.

In [ ]:
# Copy into Drive so a new runtime can pick them up.
shutil.copy2(MODEL_PATH, MODEL_PATH)                       # already there
tokenizer_path = CharTokenizer.default_path(MODEL_PATH)
for f in (MODEL_PATH, tokenizer_path):
    print(f"{os.path.getsize(f)/1024**2:7.2f} MB  {f}")

In [ ]:
# Download to your machine (pick a folder when prompted).
try:
    from google.colab import files
    files.download(MODEL_PATH)
    files.download(CharTokenizer.default_path(MODEL_PATH))
    print("Downloaded — keep both files in the same folder.")
except Exception as e:
    print("Download skipped:", e)

## Next steps

Raise quality by changing one thing at a time:

| Want | Change |
|---|---|
| Better text | More/longer corpus. Loss is data-limited, not size-limited. |
| Finer detail | Larger `block_size` (512+) so it sees more context |
| More capacity | `PRESET = "medium"`, and raise `max_iters` |
| Less overfitting | Raise `dropout`, lower `n_layer` |
| Faster epochs | Lower `block_size`, raise `batch_size` |

To use the model on your own machine afterwards, see the README — the
checkpoint is plain CPU tensors and loads anywhere:

```bash
pip install -r requirements.txt
python setup.py generate --model model.pth --prompt "The"
```

If a run got cut off, set `RUN_NAME` to a new value and retrain from scratch —
or keep the same name and pass `--resume` to continue.